# 3.1 · 探索性数据分析 / Exploratory Data Analysis

> **课程定位 / Where this fits**
> **Part 3 第 1 课**。Part 0-2 备齐了工具（pandas/可视化）和统计武器（假设检验/相关）；从这一 Part 起，**对真实脏数据动手**。EDA 是任何建模的第一步——**不做 EDA 直接建模 = 蒙眼开车**。
> Part 3, lesson 1. We now touch real dirty data. EDA is the first step of any modeling — skipping it is driving blind.

> 📐 **符号约定**（见 [`NOTATION.md`](../NOTATION.md)）：$n$ 样本数，$d$ 特征数。

> 💡 **面试相关 / Interview-relevant**
> - "拿到一份新数据你会做什么" ★★★★★（开放题，EDA 流程是标准答案）
> - "怎么发现数据质量问题" ★★★★
> - "单变量/双变量/多变量分析各看什么" ★★★

---

## 学习目标 / Learning Objectives
1. 形成一套**可复用的 EDA 检查清单**（结构 → 质量 → 单变量 → 双变量 → 多变量）。
2. 区分**数值/类别/时间**变量并对每类用对的图。
3. 用 EDA **提出假设**（再交给 Part 2 的检验回答）。
4. 在 Titanic 上产出一份**能讲故事**的 EDA 报告。

## 目录 / TOC
1. [EDA 的五步检查清单 ⭐](#1)
2. [🛳 数据集与第一眼](#2)
3. [结构与质量审计](#3)
4. [单变量分析 / Univariate](#4)
5. [双变量分析 / Bivariate](#5)
6. [多变量分析 / Multivariate](#6)
7. [EDA → 假设](#7)
8. [小结](#8)


<a id="1"></a>
## 1. EDA 的五步检查清单 ⭐ / The Five-Step Checklist

```
1. 结构 Structure    : shape, dtypes, 前几行 — "这是什么形状的数据"
2. 质量 Quality      : 缺失 / 重复 / 常量列 / 异常类型 — "能信吗"
3. 单变量 Univariate : 每列自己的分布 — "每个变量长什么样"
4. 双变量 Bivariate  : 特征 vs 目标, 特征 vs 特征 — "什么和什么有关"
5. 多变量 Multivariate: 相关矩阵, 交互, 降维 — "整体结构是什么"
```

**这套清单就是"拿到新数据你会做什么"的满分答案**。EDA 不是随便画图，是**有结构的侦查**。
This checklist IS the full-credit answer to "what do you do with a new dataset". EDA is structured detective work, not random plotting.


<a id="2"></a>
## 2. 🛳 数据集与第一眼 / Dataset & First Glance

复用 0.3 节的 Titanic（891 名乘客）。0.3 我们学的是 pandas **机制**；这次学的是 **EDA 方法论**——同一份数据，不同的视角。
Reusing Titanic from 0.3. There we learned pandas mechanics; here, EDA methodology.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

df = sns.load_dataset("titanic")
print(f"shape: {df.shape}")
df.head(3)


<a id="3"></a>
## 3. 结构与质量审计 / Structure & Quality Audit

**第一眼必跑的诊断**——一个函数封装全部质量检查：


In [ ]:
def quality_report(df):
    rep = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean()*100).round(1),
        "n_unique": df.nunique(),
        "pct_unique": (df.nunique()/len(df)*100).round(1),
    })
    # 标记可疑列 / flag suspicious columns
    rep["flag"] = ""
    rep.loc[rep.n_unique == 1, "flag"] += "常量;"          # constant column
    rep.loc[rep.n_unique == len(df), "flag"] += "全唯一(可能是ID);"  # ID-like
    rep.loc[rep.pct_missing > 50, "flag"] += "高缺失;"      # high missing
    return rep

quality_report(df)


**一表看出全部质量问题**：
- `deck` 缺失 77% → 大概率要丢（3.2 节细讲）
- `age` 缺失 20% → 必须处理
- `alive` 是 `survived` 的字符串副本（**数据泄漏**，3.9 节正题）→ 建模必删

> 💡 把 `quality_report` 存进你的工具箱——每个新项目第一行就跑它。
> Keep `quality_report` in your toolkit — run it first on every new dataset.


In [ ]:
# 重复行检查 / Duplicate check
print(f"完全重复的行: {df.duplicated().sum()}")
# 关键列重复 (可能是数据录入问题) / dup on key columns
print(f"重复行 (忽略可能不同的字段): 用业务主键检查, Titanic 无显式 ID")


<a id="4"></a>
## 4. 单变量分析 / Univariate Analysis

**每种类型的变量用对的图**：

| 类型 | 看什么 | 图 |
|---|---|---|
| 数值 | 分布形状/偏度/异常 | histogram + boxplot |
| 类别 | 各类频数/不平衡 | countplot |
| 目标 | 类别平衡 | countplot / 比例 |


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# 数值: age (有缺失, 单峰右偏) / numeric
sns.histplot(df["age"].dropna(), bins=30, kde=True, ax=axes[0,0])
axes[0,0].set_title(f"age (skew={st.skew(df['age'].dropna()):.2f})")

# 数值: fare (极右偏 — 注意!) / numeric, very skewed
sns.histplot(df["fare"], bins=40, ax=axes[0,1])
axes[0,1].set_title(f"fare (skew={st.skew(df['fare']):.2f}) — 极右偏")

# fare 的 log 变换 (右偏数据的标准动作) / log transform
sns.histplot(np.log1p(df["fare"]), bins=40, ax=axes[0,2])
axes[0,2].set_title("log1p(fare) — 接近对称了")

# 类别: pclass / categorical
sns.countplot(data=df, x="pclass", ax=axes[1,0])
axes[1,0].set_title("pclass (三等舱最多)")

# 类别: embarked / categorical
sns.countplot(data=df, x="embarked", ax=axes[1,1])
axes[1,1].set_title("embarked")

# 目标: survived (类别平衡检查) / target balance
surv = df["survived"].value_counts(normalize=True)
axes[1,2].bar(["died(0)", "survived(1)"], surv.values, color=["#d62728","#2ca02c"])
axes[1,2].set_title(f"survived: {surv[1]:.0%} 生还 (略不平衡)")
plt.tight_layout(); plt.show()


**单变量阶段的收获**：
- `fare` 极右偏（skew 4.8）→ 建模前考虑 log 变换（3.6 节）
- `survived` 38% / 62% → 轻度不平衡（Part 3.11 会处理更极端的）
- `pclass` 三等舱占多数 → 后面看它和生还的关系


<a id="5"></a>
## 5. 双变量分析 / Bivariate — 特征 vs 目标

**这是 EDA 最有价值的部分**：哪些特征和目标有关？关系是什么方向？


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# 类别特征 vs 目标: 生还率 (barplot 自动算均值) / categorical vs target
for ax, col in zip(axes[0], ["sex", "pclass", "embarked"]):
    sns.barplot(data=df, x=col, y="survived", ax=ax)
    ax.set_title(f"survival rate by {col}")
    ax.set_ylim(0, 1)

# 数值特征 vs 目标: 按 survived 分组看分布 / numeric vs target
sns.boxplot(data=df, x="survived", y="age", ax=axes[1,0])
axes[1,0].set_title("age by survived")
sns.boxplot(data=df, x="survived", y="fare", ax=axes[1,1])
axes[1,1].set_yscale("log"); axes[1,1].set_title("fare by survived (log)")

# 派生特征: 家庭规模 vs 生还 / engineered feature
df2 = df.assign(family_size=df.sibsp + df.parch + 1)
sns.barplot(data=df2, x="family_size", y="survived", ax=axes[1,2])
axes[1,2].set_title("survival by family_size (非单调!)")
plt.tight_layout(); plt.show()


**双变量阶段的发现**（与 0.3 节 EDA 呼应，但这次用图系统化）：
- **性别**是最强信号：女性 74% vs 男性 19%
- **舱等**强相关：一等 63% vs 三等 24%
- **票价**高的更易生还（和舱等共线）
- **家庭规模非单调**：单身和大家庭都低，3-4 人最高 → 提示需要**分箱**而非线性处理（3.6 节）


<a id="6"></a>
## 6. 多变量分析 / Multivariate

**看整体结构**：相关矩阵 + 交互效应。


In [ ]:
# 相关矩阵 (数值列) / correlation matrix
num = df[["survived","pclass","age","sibsp","parch","fare"]]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            center=0, ax=axes[0])
axes[0].set_title("correlation matrix")

# 交互: 舱等 × 性别 → 生还率 (热力图最直观) / interaction effect
pivot = df.pivot_table(values="survived", index="pclass", columns="sex",
                       aggfunc="mean", observed=True)
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("survival: pclass × sex (强交互!)")
plt.tight_layout(); plt.show()


**多变量阶段的关键发现 = 交互效应**：
- 一等舱女性生还率 **97%**，三等舱男性仅 **14%** —— 差距 7 倍
- 单看性别或单看舱等都不完整；**两者交互**才是真相
- 这提示：树模型（天然抓交互）可能比线性模型更适合（Part 4-5 验证）

> 💡 **相关矩阵的局限**：只抓**线性**关系（2.1 节教训）。`pclass` 和 `survived` 相关 -0.34 看似弱，但配合 `sex` 后预测力极强——**相关低 ≠ 没用**。
> Correlation only catches linear association; a low correlation feature can be powerful in interaction.


<a id="7"></a>
## 7. EDA → 假设 / From EDA to Hypotheses

**EDA 的终点是假设清单**，交给后续步骤：

| EDA 观察 | 提出的假设 / 行动 |
|---|---|
| fare 极右偏 | → log 变换（3.6）|
| age 缺 20% | → 按 pclass 分组中位数填补（3.2）|
| family_size 非单调 | → 分箱成 alone/small/large（3.6）|
| pclass × sex 强交互 | → 树模型 or 显式交互特征 |
| alive ≡ survived | → 删除（数据泄漏，3.9）|
| deck 缺 77% | → 删除或"是否有舱位"二值化 |

**EDA 不下结论，EDA 生成待办**。每个假设后续用统计检验（Part 2）或模型验证。
EDA doesn't conclude — it generates a to-do list. Each hypothesis gets tested later.


In [ ]:
# 用 Part 2 的检验验证一个 EDA 假设: "性别影响生还" / Test an EDA hypothesis
ct = pd.crosstab(df["sex"], df["survived"])
chi2, p, dof, _ = st.chi2_contingency(ct)
print(f"性别 × 生还 卡方检验: chi2={chi2:.1f}, p={p:.2e}")
print(f"→ EDA 看到的性别差异在统计上极显著 (p < 0.001), 不是噪声")


<a id="8"></a>
## 8. 小结 / Summary

```
EDA 五步清单 ⭐:
  结构(shape/dtype) → 质量(缺失/重复/常量/泄漏) → 单变量 → 双变量 → 多变量
quality_report() 一函数搞定质量审计 — 进工具箱
单变量: 数值看分布(偏度→变换), 类别看平衡
双变量: 特征 vs 目标 (最有价值)
多变量: 相关矩阵(只抓线性) + 交互(热力图)
终点: 假设清单 → 后续步骤
```

### 💡 面试速查
1. **"拿到新数据做什么"** = 背五步清单
2. **EDA 生成假设, 不下结论**
3. **相关低 ≠ 没用**（交互效应）
4. **极右偏数据**（fare/收入/时长）→ log 变换

### 下一节
**3.2 缺失值处理**——EDA 发现的 age 20% / deck 77% 缺失怎么办？MCAR/MAR/MNAR 机制 + 各种填补方法。
